In [2]:
import ee
ee.Initialize()

/home/wmlegion/miniconda3/envs/agri_land_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
import json
import pandas as pd

def get_soil_profile_area(lat, lon, area_meters=56):
    # Definimos el área: un buffer de 56 metros alrededor del punto = ~1 hectárea
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()
    
    properties = {
        'phh2o': 'projects/soilgrids-isric/phh2o_mean',
        'soc': 'projects/soilgrids-isric/soc_mean',
        'clay': 'projects/soilgrids-isric/clay_mean',
        'sand': 'projects/soilgrids-isric/sand_mean',
        'silt': 'projects/soilgrids-isric/silt_mean',
        'bdod': 'projects/soilgrids-isric/bdod_mean',
        'cec': 'projects/soilgrids-isric/cec_mean'
    }
    
    depths = ['0-5cm', '5-15cm', '15-30cm', '30-60cm', '60-100cm']  # Profundidades en centímetros 60-100cm_mean
    
    results = {}
    
    for prop, collection_id in properties.items():
        results[prop] = {}
        try:
            image = ee.Image(collection_id)
            
            for depth in depths:
                band_name = f"{prop}_{depth}_mean"
                
                # Usamos Reducer.mean() sobre la región completa para cada profundidad
                val = image.reduceRegion(
                    reducer=ee.Reducer.mean(), 
                    geometry=region,
                    scale=250, # Resolución nativa de SoilGrids
                    bestEffort=True
                ).get(band_name)
                
                results[prop][depth] = val.getInfo()
                
        except Exception as e:
            results[prop] = f"Error: {e}"
            
    return results

# Probamos con el área de 1 hectárea
# Finca Matanza 7.300921,-73.009794
soil_data = get_soil_profile_area(7.3297, -73.1867)
# print("Perfil de suelo completo por profundidades (1ha):")
# print(json.dumps(soil_data, indent=4))

# Assuming 'soil_data' is your nested dictionary result
df = pd.DataFrame(soil_data).T
df.index.name = 'property'
df.reset_index(inplace=True)
print(df.head(10))

# Save to CSV
df.to_csv("../databases/soil_profile_data_02.csv", index=False)
print("CSV export successful!")

  property       0-5cm      5-15cm     15-30cm     30-60cm    60-100cm
0    phh2o   52.509804   53.254902   53.000000   53.254902   54.254902
1      soc  568.901961  387.764706  222.705882  161.568627  168.019608
2     clay  344.470588  351.941176  375.215686  427.686275  418.431373
3     sand  312.019608  314.235294  311.470588  280.960784  276.450980
4     silt  344.254902  334.568627  313.568627  291.098039  305.117647
5     bdod   99.254902  102.509804  107.764706  112.509804  117.254902
6      cec  186.901961  184.803922  168.294118  162.784314  159.039216
CSV export successful!


In [ ]:
import json
from datetime import datetime
from pathlib import Path
import pandas as pd
import math


def get_soil_profile_area(lat, lon, area_meters=56):
    # Definimos el área: un buffer de 56 metros alrededor del punto = ~1 hectárea
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()

    properties = {
        'phh2o': 'projects/soilgrids-isric/phh2o_mean',
        'soc': 'projects/soilgrids-isric/soc_mean',
        'clay': 'projects/soilgrids-isric/clay_mean',
        'sand': 'projects/soilgrids-isric/sand_mean',
        'silt': 'projects/soilgrids-isric/silt_mean',
        'bdod': 'projects/soilgrids-isric/bdod_mean',
        'cec': 'projects/soilgrids-isric/cec_mean'
    }
    depths = ['0-5cm', '5-15cm', '15-30cm', '30-60cm', '60-100cm']  # Profundidades en centímetros

    results = {}
    for prop, collection_id in properties.items():
        results[prop] = {}
        try:
            image = ee.Image(collection_id)
            for depth in depths:
                band_name = f"{prop}_{depth}_mean"
                # Usamos Reducer.mean() sobre la región completa para cada profundidad
                val = image.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=region,
                    scale=250,  # Resolución nativa de SoilGrids
                    bestEffort=True
                ).get(band_name)
                results[prop][depth] = val.getInfo()
        except Exception as e:
            results[prop] = f"Error: {e}"

    return results


def save_soil_profile(soil_data, out_prefix="soil_profile_data", output_dir="../databases"):
    """
    Guarda el DataFrame del perfil de suelo con timestamp en el nombre,
    igual que los mapas: {out_prefix}-vYYMMDDHHMMSS.csv
    """
    df = pd.DataFrame(soil_data).T
    df.index.name = 'property'
    df.reset_index(inplace=True)

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path, df


if __name__ == "__main__":
    
# El Playon         --||     7.4584221918243045,    -73.222052853104
# Finca Matanza     --||     7.300921,              -73.009794
# Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
# Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223

    ha = 2
    area_meters = (math.sqrt(ha*10000))/2
    soil_data = get_soil_profile_area(7.4584221918243045,-73.222052853104, area_meters)

    out_path, df = save_soil_profile(soil_data)
    print(df.head(10))

CSV guardado en ../databases/soil_profile_data-v260805173352.csv (7x6)
  property       0-5cm      5-15cm     15-30cm     30-60cm    60-100cm
0    phh2o   53.000000   52.000000   53.000000   54.000000   55.000000
1      soc  592.142857  492.571429  257.714286  235.428571  279.142857
2     clay  357.000000  362.000000  411.000000  422.000000  433.000000
3     sand  336.000000  330.000000  302.000000  290.000000  277.000000
4     silt  307.000000  307.000000  287.000000  289.000000  290.000000
5     bdod   95.000000   98.000000  102.000000  106.000000  111.000000
6      cec  196.000000  179.000000  163.000000  152.000000  146.000000


In [4]:
import json
from datetime import datetime
from pathlib import Path
import pandas as pd
import math
import ee
ee.Initialize()

def get_soil_profile_area(lat, lon, area_meters=56):
    # Definimos el área: un buffer alrededor del punto
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()

    properties = {
        'phh2o': 'projects/soilgrids-isric/phh2o_mean',
        'soc': 'projects/soilgrids-isric/soc_mean',
        'clay': 'projects/soilgrids-isric/clay_mean',
        'sand': 'projects/soilgrids-isric/sand_mean',
        'silt': 'projects/soilgrids-isric/silt_mean',
        'bdod': 'projects/soilgrids-isric/bdod_mean',
        'cec': 'projects/soilgrids-isric/cec_mean'
    }
    depths = ['0-5cm', '5-15cm', '15-30cm', '30-60cm', '60-100cm']

    results = {}
    for prop, collection_id in properties.items():
        results[prop] = {}
        try:
            image = ee.Image(collection_id)
            for depth in depths:
                band_name = f"{prop}_{depth}_mean"
                val = image.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=region,
                    scale=250,
                    bestEffort=True
                ).get(band_name)
                results[prop][depth] = val.getInfo()
        except Exception as e:
            results[prop] = f"Error: {e}"

    return results


def calculate_saxton_rawls_safe(soil_data):
    """
    Versión robusta y blindada del cálculo de Saxton-Rawls (1986).
    Garantiza que las entradas estén en fracciones reales (0 a 100%) 
    y acota matemáticamente los resultados a límites físicos reales de los suelos.
    """
    depths = ['0-5cm', '5-15cm', '15-30cm', '30-60cm', '60-100cm']
    
    # Espesores de cada capa en centímetros
    layer_thicknesses = {
        '0-5cm': 5,
        '5-15cm': 10,
        '15-30cm': 15,
        '30-60cm': 30,
        '60-100cm': 40
    }
    
    hydric_results = {
        'Field_Capacity_CC_%': {},
        'Wilting_Point_PMP_%': {},
        'Available_Water_Capacity_AWC_%': {},
        'AWC_layer_mm': {}
    }

    for depth in depths:
        try:
            # 1. Extracción y conversión segura a porcentajes (0-100)
            # SoilGrids g/kg / 10 = %
            raw_clay = float(soil_data['clay'][depth])
            raw_sand = float(soil_data['sand'][depth])
            raw_silt = float(soil_data['silt'][depth])
            
            # Si los valores vienen en g/kg (>100), los llevamos a porcentaje (0-100)
            c_clay = raw_clay / 10.0 if raw_clay > 100 else raw_clay
            c_sand = raw_sand / 10.0 if raw_sand > 100 else raw_sand
            c_silt = raw_silt / 10.0 if raw_silt > 100 else raw_silt
            
            # Normalizar para que sumen 100% exacto por seguridad textural
            total_text = c_clay + c_sand + c_silt
            if total_text > 0:
                c_clay = (c_clay / total_text) * 100
                c_sand = (c_sand / total_text) * 100
                c_silt = (c_silt / total_text) * 100

            # 2. Materia Orgánica (MO %)
            # soil_data['soc'] en dg/kg -> dividir por 100 para % de Carbono Orgánico, luego * 1.724 para MO
            raw_soc = float(soil_data['soc'][depth])
            c_soc_pct = raw_soc / 100.0 if raw_soc > 100 else raw_soc / 10.0
            c_om = c_soc_pct * 1.724
            # Límite realista de MO en suelos minerales (usualmente 0.1% a 15%)
            c_om = max(0.1, min(c_om, 15.0))

            # 3. Densidad Aparente (BD in g/cm3)
            # SoilGrids bdod viene en cg/cm3 -> dividir por 100
            raw_bd = float(soil_data['bdod'][depth])
            c_bd = raw_bd / 100.0 if raw_bd > 10 else raw_bd
            # Límite físico de densidad aparente (0.8 a 2.0 g/cm3)
            c_bd = max(0.8, min(c_bd, 2.0))

            # Fracciones unitarias (0.0 a 1.0) para las fórmulas de Saxton
            S = c_sand / 100.0
            C = c_clay / 100.0
            OM = c_om / 100.0 # O en algunas adaptaciones de Saxton se usa en % directo, 
                             # pero mantengamos la proporción estándar. 
                             # Nota: La fórmula original de S&R usa OM en porcentaje directo (ej. 2.5 para 2.5%)
            # Corrigiendo OM a porcentaje directo (0 - 15) para la ecuación clásica:
            OM_pct = c_om

            # Ecuaciones Clásicas de Saxton & Rawls (1986)
            # WP (Punto de Marchitez - 1500 kPa)
            wp_33 = (-0.024 * c_sand) + (0.487 * c_clay) + (0.006 * OM_pct) \
                    - (0.005 * c_sand * OM_pct) + (0.013 * c_clay * OM_pct) + (0.068 * c_sand * c_clay) + 0.031
            wp = wp_33 + (0.14 * wp_33) - 0.02

            # FC (Capacidad de Campo - 33 kPa)
            fc_33 = (-0.251 * c_sand) + (0.195 * c_clay) + (0.011 * OM_pct) \
                    + (0.006 * c_sand * OM_pct) - (0.027 * c_clay * OM_pct) + (0.045 * c_sand * c_clay) + 0.297
            fc = fc_33 + (1.28 * (fc_33 ** 2)) - (0.38 * fc_33) - (0.03 * c_sand * fc_33) + 0.02

            # Ajuste por Densidad Aparente (Bulk Density adjustment)
            # Si la densidad difiere de la estándar de calibración (apx 1.35)
            # Omitimos sobreajustes extremos aplicando topes lógicos volumétricos:
            
            # Convertir a porcentajes volumétricos lógicos (0% a 60%)
            fc_pct = max(5.0, min(fc * 100, 55.0))
            wp_pct = max(2.0, min(wp * 100, 40.0))
            
            if fc_pct < wp_pct: # Consistencia física: CC siempre debe ser mayor a PMP
                fc_pct = wp_pct + 5.0

            awc_pct = fc_pct - wp_pct
            
            # Lámina de agua disponible por capa en mm = (AWC % / 100) * espesor de capa en mm
            thickness_mm = layer_thicknesses[depth] * 10
            awc_mm = (awc_pct / 100.0) * thickness_mm

            hydric_results['Field_Capacity_CC_%'][depth] = round(fc_pct, 2)
            hydric_results['Wilting_Point_PMP_%'][depth] = round(wp_pct, 2)
            hydric_results['Available_Water_Capacity_AWC_%'][depth] = round(awc_pct, 2)
            hydric_results['AWC_layer_mm'][depth] = round(awc_mm, 2)

        except Exception as e:
            print(f"Error procesando Saxton-Rawls en profundidad {depth}: {e}")

    df_hydric = pd.DataFrame(hydric_results)
    return df_hydric


def save_soil_profile(soil_data, out_prefix="soil_profile_data", output_dir="../databases"):
    df = pd.DataFrame(soil_data).T
    df.index.name = 'property'
    df.reset_index(inplace=True)

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV de suelo guardado en {out_path}")
    return out_path, df


if __name__ == "__main__":
    ha = 2
    area_meters = (math.sqrt(ha * 10000)) / 2
    
    # El Playon         --||     7.4584221918243045,    -73.222052853104
    # Finca Matanza     --||     7.300921,              -73.009794
    # Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
    # Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223
    
    
    # Coordenadas de prueba (El Playón)
    lat, lon = -19.689669877950884, 147.22717515914223
    
    print(f"📥 Extrayendo datos de suelo para lat: {lat}, lon: {lon}...")
    soil_data = get_soil_profile_area(lat, lon, area_meters)

    # Guardar propiedades originales
    out_path, df = save_soil_profile(soil_data)
    print("\n--- PROPIEDADES ORIGINALES (SoilGrids) ---")
    print(df.head(10))

    # Calcular Saxton-Rawls
    print("\n💧 Calculando propiedades hídricas (Saxton-Rawls)...")
    df_hydric = calculate_saxton_rawls_safe(soil_data)
    print(df_hydric)
    
    # Lámina total de agua disponible en el perfil (0 - 100 cm)
    total_awc_mm = df_hydric['AWC_layer_mm'].sum()
    print(f"\n🌱 Lámina Total de Agua Disponible (AWC) en el perfil radicular (0-100cm): {total_awc_mm:.2f} mm")

📥 Extrayendo datos de suelo para lat: -19.689669877950884, lon: 147.22717515914223...
CSV de suelo guardado en ../databases/soil_profile_data-v260806185111.csv

--- PROPIEDADES ORIGINALES (SoilGrids) ---
  property       0-5cm      5-15cm     15-30cm     30-60cm    60-100cm
0    phh2o   63.000000   65.000000   69.024390   77.000000   80.000000
1      soc  185.000000  120.000000   98.000000  123.000000  222.000000
2     clay  316.304878  289.146341  454.219512  458.426829  456.378049
3     sand  427.219512  446.292683  330.500000  328.329268  334.329268
4     silt  256.475610  264.560976  215.280488  212.256098  208.304878
5     bdod  129.000000  132.000000  133.000000  136.000000  138.000000
6      cec  127.256098  135.085366  135.121951  141.146341  145.231707

💧 Calculando propiedades hídricas (Saxton-Rawls)...
          Field_Capacity_CC_%  Wilting_Point_PMP_%  \
0-5cm                    55.0                 40.0   
5-15cm                   55.0                 40.0   
15-30cm      

In [2]:
"""
Extrae propiedades de suelo (SoilGrids-ISRIC) para un área alrededor de
un punto, y calcula sus propiedades hídricas (ver soil_hydraulics.py
para el detalle del cálculo y cómo leer el resultado).
"""
from datetime import datetime
from pathlib import Path
import math
import pandas as pd
import ee

from soil_hydraulics import DEPTHS, calculate_hydraulic_properties


def get_soil_profile_area(lat, lon, area_meters=56):
    """
    Extrae phh2o, soc, clay, sand, silt, bdod, cec para 5 profundidades
    estándar, promediados sobre un área alrededor del punto.
    """
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()

    properties = {
        'phh2o': 'projects/soilgrids-isric/phh2o_mean',
        'soc': 'projects/soilgrids-isric/soc_mean',
        'clay': 'projects/soilgrids-isric/clay_mean',
        'sand': 'projects/soilgrids-isric/sand_mean',
        'silt': 'projects/soilgrids-isric/silt_mean',
        'bdod': 'projects/soilgrids-isric/bdod_mean',
        'cec': 'projects/soilgrids-isric/cec_mean',
    }

    results = {}
    for prop, collection_id in properties.items():
        results[prop] = {}
        image = ee.Image(collection_id)
        for depth in DEPTHS:
            band_name = f"{prop}_{depth}_mean"
            val = image.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=region,
                scale=250,
                bestEffort=True
            ).get(band_name)
            results[prop][depth] = val.getInfo()

    return results


def save_soil_profile(soil_data, out_prefix="soil_profile_data", output_dir="../databases"):
    """Guarda las propiedades crudas de SoilGrids con timestamp en el nombre."""
    df = pd.DataFrame(soil_data).T
    df.index.name = 'property'
    df.reset_index(inplace=True)

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path, df


def save_hydraulic_profile(df_hydric, out_prefix="soil_hydraulic_data", output_dir="../databases"):
    """Guarda las propiedades hídricas calculadas con timestamp en el nombre."""
    df = df_hydric.reset_index().rename(columns={'index': 'depth'})

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    ha = 2
    area_meters = math.sqrt(ha * 10000) / 2

    # El Playon         --||     7.4584221918243045,    -73.222052853104
    # Finca Matanza     --||     7.300921,              -73.009794
    # Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
    # Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223
    
    lat, lon = -19.689669877950884, 147.22717515914223

    soil_data = get_soil_profile_area(lat, lon, area_meters)
    save_soil_profile(soil_data)

    df_hydric = calculate_hydraulic_properties(soil_data)
    save_hydraulic_profile(df_hydric)
    print(df_hydric)

    total_awc_mm = df_hydric['AWC_layer_mm'].sum()
    print(f"\nLámina total de agua disponible (0-100cm): {total_awc_mm:.2f} mm")

CSV guardado en ../databases/soil_profile_data-v260806220514.csv (7x6)
CSV guardado en ../databases/soil_hydraulic_data-v260806220514.csv (5x5)
          Field_Capacity_CC_%  Wilting_Point_PMP_%  \
0-5cm                    55.0                 40.0   
5-15cm                   55.0                 40.0   
15-30cm                  55.0                 40.0   
30-60cm                  55.0                 40.0   
60-100cm                 55.0                 40.0   

          Available_Water_Capacity_AWC_%  AWC_layer_mm  
0-5cm                               15.0           7.5  
5-15cm                              15.0          15.0  
15-30cm                             15.0          22.5  
30-60cm                             15.0          45.0  
60-100cm                            15.0          60.0  

Lámina total de agua disponible (0-100cm): 150.00 mm
